# PyTorch 零基础 4/6：Autograd、损失与优化器

这是第 1～41 课 ASR 主线之前的桥梁课。先预测，再运行；看懂输出后必须改一个值验证自己的解释。

| 项目 | 内容 |
|---|---|
| 前置要求 | 完成基础 3；理解 Tensor 运算和矩阵乘法 |
| 建议投入 | 60～90 分钟，可分两次完成 |
| 核心概念 | 计算图与 requires_grad、loss 与 backward、zero_grad 与 optimizer.step |
| 完成标准 | 能解释代码、独立完成练习、从空白重写本课核心函数 |


## 课前诊断（先不要运行代码）

1. 用自己的话解释：计算图与 requires_grad。
2. 猜测 loss 与 backward 最容易出现哪一种错误。
3. 写下你对 zero_grad 与 optimizer.step 的暂时理解；不会可以明确写“不知道”。

这三题不计分，只用于留下学习前证据。


## 1. PyTorch 自动记录参数如何影响结果


In [1]:
import torch

w = torch.tensor(0.0, requires_grad=True)
x = torch.tensor(3.0)
prediction = w * x
loss = (prediction - 6.0) ** 2
loss.backward()

print("prediction:", prediction.item())
print("loss:", loss.item())
print("d(loss)/d(w):", w.grad.item())
assert w.grad.item() == -36.0


prediction: 0.0
loss: 36.0
d(loss)/d(w): -36.0


## 2. 梯度不是更新；optimizer.step 才修改参数

`backward()` 计算方向，`step()` 执行更新。梯度默认累加，因此每轮需要清零。


In [2]:
w = torch.nn.Parameter(torch.tensor(0.0))
optimizer = torch.optim.SGD([w], lr=0.05)

for step in range(6):
    prediction = w * 3.0
    loss = (prediction - 6.0) ** 2
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"step={step}, w={w.item():.4f}, loss={loss.item():.4f}")

assert abs(w.item() - 2.0) < 0.01


step=0, w=1.8000, loss=36.0000
step=1, w=1.9800, loss=0.3600
step=2, w=1.9980, loss=0.0036
step=3, w=1.9998, loss=0.0000
step=4, w=2.0000, loss=0.0000
step=5, w=2.0000, loss=0.0000


## 3. 完整的最小线性回归


In [3]:
torch.manual_seed(0)
x = torch.tensor([[-2.0], [-1.0], [0.0], [1.0], [2.0]])
y = 2 * x + 1
model = torch.nn.Linear(1, 1)
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

history = []
for step in range(80):
    prediction = model(x)
    loss = loss_fn(prediction, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    history.append(loss.item())

print("first/final loss:", history[0], history[-1])
print("learned weight/bias:", model.weight.item(), model.bias.item())
assert history[-1] < history[0] * 1e-4
assert abs(model.weight.item() - 2.0) < 0.02
assert abs(model.bias.item() - 1.0) < 0.02


first/final loss: 8.274890899658203 5.684341970784096e-15
learned weight/bias: 2.0 0.9999998807907104


## 4. 推理时不需要梯度


In [4]:
with torch.no_grad():
    test_prediction = model(torch.tensor([[4.0]]))

print("x=4 prediction:", test_prediction.item())
assert not test_prediction.requires_grad
assert abs(test_prediction.item() - 9.0) < 0.05


x=4 prediction: 9.0


## 5. 梯度审计：先看是否存在，再看是否有限


In [5]:
for name, parameter in model.named_parameters():
    assert parameter.grad is not None
    assert torch.isfinite(parameter.grad).all()
    print(name, "grad norm =", parameter.grad.norm().item())


weight grad norm = 4.7683716530855236e-08
bias grad norm = 9.536743306171047e-08


## 本课练习（保留作答区）


1. 区分 prediction、target 和 loss。
2. `backward()` 与 `optimizer.step()` 各做什么？
3. 为什么每轮要 `zero_grad()`？
4. 把学习率改成 0、0.01、1.0，预测并记录 loss 曲线。
5. 手算 `w=0,x=3,target=6` 时平方误差对 w 的梯度。
6. 从空白重写最小线性回归训练循环。
7. 故意删除 `zero_grad()`，比较参数和梯度变化。
8. 加入断言检查 loss、梯度和参数都为有限数。
9. 解释 `torch.no_grad()` 为什么用于验证或推理。
10. 把直线训练的 input/model/loss 映射到 ASR 的特征/编码器/CTC loss。


评分：每题 0～2 分。达到 16/20 可以继续；12～15 分次日重做错题；低于 12 分回看代码并从空白复现。


## 离场票与间隔复习

- [ ] 我能闭卷解释：计算图与 requires_grad、loss 与 backward、zero_grad 与 optimizer.step。
- [ ] 我能预测核心代码的 shape、dtype 或数值方向。
- [ ] 我能从空白重写至少一个函数，并通过正常、边界、错误输入测试。
- [ ] 我能说出一个“代码能运行但语义错误”的例子。

复习安排：明天闭卷回忆 5 分钟；7 天后重做第 4、7、10 题；30 天后重新构造最小实验。

下一步：基础 5：nn.Module、训练/验证模式与保存加载。
